# Lecture 04: Neural LMs and Recurrent Attention

Train a fixed-window LM, trace recurrent memory, and compute additive alignment in an RNN encoder–decoder. The notebook runs offline on CPU with supplied toy data. All practices are ungraded, with the same expectations for all students.

| Exercise | Time | Evidence |
| --- | --- | --- |
| E01 | 4 min | Context windows and model shapes |
| E02 | 6 min | Connected gradients and a tiny fitting check |
| E03 | 7 min | A recurrent state and its input/parameter derivatives |
| E04 | 7 min | Cell memory, exposure, and a direct gradient path |
| E05 | 7 min | A decoder-dependent weighted source context |

Predict first, then run the next cell. Q/K/V, scaled dot products, and causal self-attention are developed in Lecture 05.

In [ ]:
import math
import torch
import torch.nn.functional as F

torch.manual_seed(7)
print('PyTorch:', torch.__version__, '| device: CPU')

## A longer context than Lecture 03

Our invented corpus has two sentences:

```
BOS red  key opens  EOS
BOS blue key closes EOS
```

Both prefixes end in `key`, but the next token differs. A bigram model uses the same distribution after `key` in both cases. With equal counts, its best distribution on these two targets assigns probability 1/2 to each and has mean loss $\log 2$ nats. A context of two tokens can distinguish them.

Vocabulary IDs are local to this notebook. `BOS` marks a sentence start and `EOS` a sentence end. We allocate one output row per vocabulary entry, including `BOS`, although `BOS` is never a training target. No window crosses a sentence boundary, and nothing is predicted after `EOS`.

In [ ]:
vocabulary = ['BOS', 'red', 'blue', 'key', 'opens', 'closes', 'EOS']
token_to_id = {token: index for index, token in enumerate(vocabulary)}
sentences = ['BOS red key opens EOS', 'BOS blue key closes EOS']
documents = [[token_to_id[token] for token in text.split()] for text in sentences]


def make_windows(documents, context_size):
    if context_size < 1:
        raise ValueError('context_size must be positive')
    contexts, targets = [], []
    for document in documents:
        for position in range(context_size, len(document)):
            contexts.append(document[position - context_size:position])
            targets.append(document[position])
    return (torch.tensor(contexts, dtype=torch.long).reshape(-1, context_size),
            torch.tensor(targets, dtype=torch.long))


C, D, H, V = 2, 4, 8, len(vocabulary)
train_contexts, train_targets = make_windows(documents, C)
for context, target in zip(train_contexts, train_targets):
    print([vocabulary[index] for index in context], '->', vocabulary[target])

## E01 · Context windows and shapes

**4 minutes.** For `BOS red key opens EOS`, enumerate the three length-2 contexts and their targets. Then take a batch of **two** contexts with $C=2$, embedding width $d=4$, hidden width $h=8$, and vocabulary size $|V|=7$. Predict the shapes after lookup, concatenation, hidden transformation, and vocabulary projection.

The network follows the NPLM example in the [Spring notebook, §3](https://github.com/baojian/llm-26/blob/main/lecture-04-neural-lms/lecture-04-neural-lms.ipynb). It uses plain `nn.Module` and a local vocabulary. We omit the optional direct input-to-output connection from [Bengio et al. (2003), §2](https://www.jmlr.org/papers/v3/bengio03a.html).

In [ ]:
class ContextLM(torch.nn.Module):
    def __init__(self, vocab_size, context_size, width, hidden_width):
        super().__init__()
        self.context_size = context_size
        self.embedding = torch.nn.Embedding(vocab_size, width)
        self.hidden = torch.nn.Linear(context_size * width, hidden_width)
        self.output = torch.nn.Linear(hidden_width, vocab_size)

    def forward(self, ids):
        if ids.ndim != 2 or ids.shape[1] != self.context_size:
            raise ValueError('ids must have shape (batch, context_size)')
        x = self.embedding(ids).flatten(start_dim=1)
        return self.output(torch.tanh(self.hidden(x)))


torch.manual_seed(7)
model = ContextLM(V, C, D, H)
e01_ids = train_contexts[:2]
e01_lookup = model.embedding(e01_ids)
e01_flat = e01_lookup.flatten(start_dim=1)
e01_hidden = torch.tanh(model.hidden(e01_flat))
e01_shapes = {
    'ids': tuple(e01_ids.shape), 'lookup': tuple(e01_lookup.shape),
    'concatenated': tuple(e01_flat.shape), 'hidden': tuple(e01_hidden.shape),
    'logits': tuple(model(e01_ids).shape),
}
print(e01_shapes)

**Check:** `[BOS, red] -> key`, `[red, key] -> opens`, `[key, opens] -> EOS`.
Shapes: `(2, 2)`, `(2, 2, 4)`, `(2, 8)`, `(2, 8)`, `(2, 7)`.
Concatenation preserves context order. The hidden nonlinearity lets the network learn interactions between positions. Without nonlinearities, consecutive affine layers collapse to one affine map.

### Stable next-token loss

Each context has **one** next-token target: logits have shape $(B,|V|)$ and targets $(B,)$. Supply raw logits to `F.cross_entropy`. Its stable form is $\log\sum_j e^{z_j}-z_y$.

Subtracting the largest logit before exponentiation avoids overflow in softmax. `logsumexp` computes the corresponding log-normalizer stably. Large logits can produce infinity in a direct `exp`; the next cell deliberately demonstrates that failure.

In [ ]:
large_logits = torch.tensor([[1000., 1001., 999.]], dtype=torch.float64)
target = torch.tensor([1])
naive_normalizer = large_logits.exp().sum(dim=-1)
stable_loss = (torch.logsumexp(large_logits, dim=-1)
               - large_logits.gather(-1, target[:, None]).squeeze(-1)).mean()
builtin_loss = F.cross_entropy(large_logits, target)
uniform_loss = F.cross_entropy(torch.zeros(2, V), train_targets[:2]).item()
print('naive normalizer:', naive_normalizer.tolist())
print('stable / PyTorch:', stable_loss.item(), builtin_loss.item())
print('uniform baseline:', uniform_loss, '| log(V):', math.log(V))

## E02 · Repair the training loop

**6 minutes.** A student writes:

```python
loss = F.cross_entropy(model(train_contexts), train_targets).detach()
loss.backward()
```

Explain why this fails, then repair it. Where should gradients be cleared? Predict what `optimizer.step()` changes. Run the corrected loop below and inspect the six predictions.

This is a deliberately consistent batch: no identical length-2 context has conflicting targets. The debugging goal is a final mean loss below **0.03 nats** after 200 SGD updates. Achieving this says nothing yet about new text.

In [ ]:
broken_model = ContextLM(V, C, D, H)
detached_loss = F.cross_entropy(broken_model(train_contexts), train_targets).detach()
try:
    detached_loss.backward()
except RuntimeError as error:
    detached_error = str(error)
    print('Expected failure:', detached_error)

In [ ]:
torch.manual_seed(7)
model = ContextLM(V, C, D, H)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
model.train()
e02_initial_embedding = model.embedding.weight.detach().clone()
with torch.no_grad():
    e02_losses = [F.cross_entropy(model(train_contexts), train_targets).item()]
e02_gradient_norms = []
for step in range(200):
    optimizer.zero_grad(set_to_none=True)
    loss = F.cross_entropy(model(train_contexts), train_targets)
    loss.backward()
    norm = torch.sqrt(sum(parameter.grad.square().sum() for parameter in model.parameters()))
    e02_gradient_norms.append(norm.item())
    optimizer.step()
    with torch.no_grad():
        e02_losses.append(F.cross_entropy(model(train_contexts), train_targets).item())
model.eval()
with torch.no_grad():
    e02_predictions = model(train_contexts).argmax(dim=-1)
print('loss before / after:', e02_losses[0], e02_losses[-1])
print('predictions:', [vocabulary[index] for index in e02_predictions])
print('correct:', int((e02_predictions == train_targets).sum()), '/', len(train_targets))

**Check:** removing `.detach()` preserves the path from the loss to the parameters. Clear gradients before `backward()`, then update after `backward()`. Detach only the scalar you record, or use `loss.item()`.

The predictions should be `key, opens, EOS, key, closes, EOS`. The losses and gradient norms should remain finite. Cross-entropy starts near the uniform baseline and ends below 0.03 nats with the specified initialization and learning rate.

This notebook intentionally has **no held-out evaluation**. For a generalization experiment, reserve separate documents before making overlapping windows, use the same tokenizer and scoring convention, and report held-out loss separately. `model.eval()` changes training-mode modules; `torch.no_grad()` disables gradient recording. Neither creates held-out data.

## Recurrent states

A fixed-window model discards inputs outside its window. A recurrent model updates a fixed-width state, which can carry information from earlier inputs:
$$h_t=\tanh(W_x x_t+W_h h_{t-1}+b).$$
The state is an activation; the same parameters are reused at every step. The small linear recurrence below isolates the path without tanh.

## E03 · Trace a memory path

**7 minutes.** For $h_t=0.5h_{t-1}+x_t$, $h_0=0$, and inputs $(1,0,1)$, find the three states and $\partial h_3/\partial x_1$. What happens to $h_3$ when only the first input becomes 2?

In [ ]:
def linear_recurrence(inputs, coefficient):
    state = inputs.new_zeros(())
    states = []
    for value in inputs:
        state = coefficient * state + value
        states.append(state)
    return torch.stack(states)

sequence = torch.tensor([1., 0., 1.], dtype=torch.float64, requires_grad=True)
coefficient = torch.tensor(0.5, dtype=torch.float64, requires_grad=True)
recurrent_states = linear_recurrence(sequence, coefficient)
input_derivatives, shared_derivative = torch.autograd.grad(
    recurrent_states[-1], (sequence, coefficient))
changed_states = linear_recurrence(torch.tensor([2., 0., 1.], dtype=torch.float64), coefficient)
print('states:', recurrent_states.tolist())
print('input derivatives:', input_derivatives.tolist())
print('shared-weight derivative:', shared_derivative.item())
print('changed first input:', changed_states.tolist())

**Check:** states $(1,0.5,1.25)$, input derivatives $(0.25,0.5,1)$, and changed final state $1.5$. Expanding $h_3=a^2x_1+ax_2+x_3$ gives the shared-weight derivative $2ax_1+x_2=1$. A real tanh RNN also includes activation derivatives along each path.

In [ ]:
def tanh_recurrence(inputs, initial, weight_x, weight_h, bias):
    state = initial
    states = []
    for value in inputs.unbind(dim=1):
        state = torch.tanh(value @ weight_x.T + state @ weight_h.T + bias)
        states.append(state)
    return torch.stack(states, dim=1), state

torch.manual_seed(23)
rnn = torch.nn.RNN(2, 3, batch_first=True).double()
rnn_inputs = torch.randn(2, 4, 2, dtype=torch.float64)
rnn_initial = torch.randn(2, 3, dtype=torch.float64)
rnn_output, rnn_final = tanh_recurrence(
    rnn_inputs, rnn_initial, rnn.weight_ih_l0, rnn.weight_hh_l0,
    rnn.bias_ih_l0 + rnn.bias_hh_l0)
library_output, library_final = rnn(rnn_inputs, rnn_initial.unsqueeze(0))
torch.testing.assert_close(rnn_output, library_output, rtol=1e-12, atol=1e-12)
torch.testing.assert_close(rnn_final, library_final[0], rtol=1e-12, atol=1e-12)
print('tanh recurrence agrees with torch.nn.RNN:', tuple(rnn_output.shape))

## LSTM memory and gates

The common modern cell uses $c_t=f_t\odot c_{t-1}+i_t\odot\widetilde c_t$ and $h_t=o_t\odot\tanh(c_t)$. Separate sigmoid gates depend on the current input and previous hidden state. The forget gate was added after the original 1997 LSTM; these are the modern PyTorch equations. A supplied-gate example isolates the arithmetic.

## E04 · One gated memory update

**7 minutes.** Given $c_{t-1}=2$, $f_t=0.75$, $i_t=0.5$, $\widetilde c_t=-0.5$, $o_t=0.5$, find $c_t$, $h_t$, and the direct derivative to the old cell while holding the gates fixed. Compare forgetting everything ($f=0$), retaining everything ($f=1$), and hiding the result ($o=0$).

In [ ]:
def supplied_gate_update(previous, forget, input_gate, candidate, output_gate):
    cell_state = forget * previous + input_gate * candidate
    hidden = output_gate * torch.tanh(cell_state)
    return cell_state, hidden

previous_cell = torch.tensor(2., dtype=torch.float64, requires_grad=True)
next_cell, next_hidden = supplied_gate_update(previous_cell, 0.75, 0.5, -0.5, 0.5)
direct_cell_derivative, = torch.autograd.grad(next_cell, previous_cell)
print('cell / hidden / direct derivative:', next_cell.item(), next_hidden.item(), direct_cell_derivative.item())

**Check:** $c_t=1.25$, $h_t\approx0.42414182$, and direct derivative $0.75$. With $f=0$, the cell becomes $-0.25$; with $f=1$, it becomes $1.75$. Setting $o=0$ hides the cell from the output while preserving its stored value. These local statements hold the gates fixed; full recurrent gradients include other paths.

In [ ]:
def explicit_lstm_cell(inputs, hidden, previous, cell):
    gates = (inputs @ cell.weight_ih.T + cell.bias_ih
             + hidden @ cell.weight_hh.T + cell.bias_hh)
    input_gate, forget, candidate, output_gate = gates.chunk(4, dim=-1)
    return supplied_gate_update(previous, forget.sigmoid(), input_gate.sigmoid(),
                                candidate.tanh(), output_gate.sigmoid())

torch.manual_seed(29)
lstm_cell = torch.nn.LSTMCell(2, 3).double()
lstm_input = torch.randn(2, 2, dtype=torch.float64)
lstm_hidden = torch.randn(2, 3, dtype=torch.float64)
lstm_previous = torch.randn(2, 3, dtype=torch.float64)
explicit_cell, explicit_hidden = explicit_lstm_cell(lstm_input, lstm_hidden, lstm_previous, lstm_cell)
library_hidden, library_cell = lstm_cell(lstm_input, (lstm_hidden, lstm_previous))
torch.testing.assert_close(explicit_cell, library_cell, rtol=1e-12, atol=1e-12)
torch.testing.assert_close(explicit_hidden, library_hidden, rtol=1e-12, atol=1e-12)
print('complete cell agrees with torch.nn.LSTMCell')

## Attention in an RNN encoder–decoder

[Bahdanau, Cho, and Bengio](https://baojian.github.io/llm-26-fall/papers/2015-iclr-bahdanau-neural-machine-translation-align-translate.pdf) introduced additive soft alignment for neural machine translation (2014 preprint, ICLR 2015). A bidirectional gated RNN produces source annotations $h_j$. The previous decoder state $s_{t-1}$ determines a new context:
$$e_{t,j}=v_a^\top\tanh(W_a s_{t-1}+U_a h_j),\quad
\alpha_{t,j}=\mathrm{softmax}_j(e_{t,j}),\quad c_t=\sum_j\alpha_{t,j}h_j.$$
The decoder uses that context to update its state and predict the next target word. The paper uses GRUs, not LSTM cells. All source words are available before decoding, while future target words are unavailable. Source and target positions are different indices; no target-index triangular mask is applied to these source annotations.

This is a foundational NMT attention paper, not the first attention mechanism of every kind. Its §6.1 discusses earlier handwriting alignment in [Graves (2013)](https://arxiv.org/abs/1308.0850).

## E05 · A different context at each target step

**7 minutes.** Use $h_1=(1,2)$ and $h_2=(3,0)$. The supplied alignment scores are $(0,\log 3)$. Calculate the weights and context, then repeat for $(\log 3,0)$ at another target step. These are invented values for a calculation, not trained translation outputs.

In [ ]:
annotations = torch.tensor([[1., 2.], [3., 0.]], dtype=torch.float64)
alignment_scores = torch.tensor([0., math.log(3)], dtype=torch.float64)
alignment_weights = alignment_scores.softmax(dim=0)
alignment_context = alignment_weights @ annotations
next_alignment_weights = alignment_scores.flip(0).softmax(dim=0)
next_alignment_context = next_alignment_weights @ annotations
print('first weights / context:', alignment_weights.tolist(), alignment_context.tolist())
print('next weights / context:', next_alignment_weights.tolist(), next_alignment_context.tolist())

**Check:** the first weights are $(1/4,3/4)$ and the context is $(2.5,0.5)$. The second weights are $(3/4,1/4)$ and context $(1.5,1.5)$. Both contexts use the same source annotations. Changing the decoder-dependent scores changes which source information is emphasized.

In [ ]:
def additive_attention(previous_state, source, weight_s, weight_h, vector):
    scores = torch.tanh(previous_state @ weight_s.T + source @ weight_h.T) @ vector
    weights = scores.softmax(dim=0)
    return scores, weights, weights @ source

# Explicit toy matrices; this demonstrates the computation, not a trained translator.
decoder_state = torch.tensor([0.4, -0.2], dtype=torch.float64, requires_grad=True)
source_states = annotations.clone().requires_grad_()
weight_s = torch.tensor([[0.5, 0.1], [-0.3, 0.7]], dtype=torch.float64, requires_grad=True)
weight_h = torch.tensor([[0.2, -0.4], [0.6, 0.3]], dtype=torch.float64, requires_grad=True)
alignment_vector = torch.tensor([1., -0.5], dtype=torch.float64, requires_grad=True)
additive_inputs = (decoder_state, source_states, weight_s, weight_h, alignment_vector)
additive_scores, additive_weights, additive_context = additive_attention(*additive_inputs)
additive_gradients = torch.autograd.grad(additive_context.square().sum(), additive_inputs)
print('additive scores / weights / context:', additive_scores.tolist(), additive_weights.tolist(), additive_context.tolist())
print('gradient norms:', [g.norm().item() for g in additive_gradients])

## Readings and continuation

- [Bengio et al. (2003), §2](https://www.jmlr.org/papers/v3/bengio03a.html): the fixed-window NPLM.
- [Pascanu et al. (2013), §2](https://proceedings.mlr.press/v28/pascanu13.html): recurrent gradient paths.
- [Bahdanau et al., §§2–3 and Appendix A.1.2](https://baojian.github.io/llm-26-fall/papers/2015-iclr-bahdanau-neural-machine-translation-align-translate.pdf): additive alignment and the recurrent decoder.
- [Graves (2013)](https://arxiv.org/abs/1308.0850): generation and earlier handwriting alignment.
- [Modern LSTM equations](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html).
- [Optional reading](https://baojian.github.io/llm-26-fall/slides/lecture-04/optional-reading.md): historical scope and longer recurrent experiments.

Lecture 05 develops Q/K/V, scaled dot products, causal self-attention, and a Transformer. No Transformer or translation benchmark is trained here.